# AI Handwriting Generator — Colab Training

**Before running:** Runtime → Change runtime type → **T4 GPU**

### What this trains
`CharNet`: character index → 24 Bézier control points learned from real handwriting.
No writer selection, no image input — the model learns the average stroke shape
for every character (A-Z, a-z, 0-9) from real handwriting samples.
Natural pen-jitter is added at generation time.

### Steps
1. Verify GPU
2. Mount Drive
3. Install deps
4. Clone repo
5. Copy Writers_pngs from Drive
6. Build Bézier label cache (skeleton extraction, ~5 min, runs once)
7. Train CharNet (fast — no image loading during training)
8. Save checkpoint to Drive + download
9. Generation demo

In [ ]:
# Cell 1: Verify GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU — go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 3: Install dependencies
!pip install opencv-contrib-python reportlab scikit-image scipy -q
print('Dependencies installed')

In [ ]:
# Cell 4: Clone / pull repo
import os

REPO      = 'AI-Powered-Handwriting-Generation-System'
REPO_PATH = f'/content/{REPO}'

if os.path.exists(REPO_PATH):
    print('Repo exists — pulling latest...')
    os.system(f'git -C {REPO_PATH} pull')
else:
    print('Cloning repo...')
    os.system(f'git clone --depth 1 https://github.com/Abdullahshaz70/{REPO}.git {REPO_PATH}')

os.chdir(REPO_PATH)
print('Working dir:', os.getcwd())

In [ ]:
# Cell 5: Copy Writers_pngs from Drive
# Upload your Writers_pngs folder to: My Drive/Writers_pngs/
import shutil, os

src  = '/content/drive/MyDrive/Writers_pngs'
dest = f'{REPO_PATH}/Data/Writers_pngs'

if os.path.exists(dest) and len(os.listdir(dest)) > 2:
    print('Writers_pngs already present — skipping copy')
elif os.path.exists(src):
    if os.path.exists(dest):
        shutil.rmtree(dest)
    shutil.copytree(src, dest)
    print('Writers_pngs copied successfully')
else:
    print('ERROR: Writers_pngs not found at', src)
    raise FileNotFoundError(src)

skip  = {'Writers_Zip', 'output_preview'}
total = 0
for d in sorted(os.listdir(dest)):
    if d in skip: continue
    p = os.path.join(dest, d)
    if os.path.isdir(p):
        n = len([f for f in os.listdir(p) if f.endswith('.png')])
        print(f'  {d}: {n} samples')
        total += n
print(f'Total: {total} samples')

In [ ]:
# Cell 6: Build Bézier label cache
# Skeletonizes every real handwriting image and extracts 24 Bézier control points.
# Saves to Data/bezier_labels.npy — only runs once, then loads from cache.
import sys
sys.path.insert(0, f'{REPO_PATH}/src')

from data import load_all_data

DATA_ROOT  = f'{REPO_PATH}/Data/Writers_pngs'
CACHE_PATH = f'{REPO_PATH}/Data/bezier_labels.npy'

records, writer_names = load_all_data(DATA_ROOT, cache_path=CACHE_PATH)
print(f'\nWriters  : {writer_names}')
print(f'Records  : {len(records)}')

In [ ]:
# Cell 7: Train CharNet
# Fast training — no image loading per batch, just char_idx → Bézier coords.
import sys, os
sys.path.insert(0, f'{REPO_PATH}/src')

import train as tr

# Override paths for Colab
tr.DATA_ROOT  = f'{REPO_PATH}/Data/Writers_pngs'
tr.CACHE_PATH = f'{REPO_PATH}/Data/bezier_labels.npy'
tr.CKPT_DIR   = f'{REPO_PATH}/checkpoints'
tr.NUM_EPOCHS = 200
tr.BATCH_SIZE = 128
tr.LR         = 1e-3

os.makedirs(tr.CKPT_DIR, exist_ok=True)
tr.main()

In [ ]:
# Cell 8: Save checkpoint to Drive + download locally
import shutil, os
from google.colab import files

ckpt_src  = f'{REPO_PATH}/checkpoints/style_net.pt'
drive_dir = '/content/drive/MyDrive/handwriting_checkpoints'
os.makedirs(drive_dir, exist_ok=True)

shutil.copy(ckpt_src, os.path.join(drive_dir, 'style_net.pt'))
print('Saved to Drive:', drive_dir)

files.download(ckpt_src)
print('\nDownload started — place style_net.pt into your local checkpoints/ folder,'
      ' then run: python app.py')

In [ ]:
# Cell 9: Generation demo — preview output for every character group
import sys
sys.path.insert(0, f'{REPO_PATH}/src')

from generate import load_model, generate_word
from PIL import Image
import matplotlib.pyplot as plt

model, ckpt, device = load_model(f'{REPO_PATH}/checkpoints/style_net.pt')
print(f'Model loaded — epoch={ckpt["epoch"]}, val_MSE={ckpt["val_loss"]:.6f}')

tests = ['Hello', 'abcdefghij', 'ABCDEFGHIJ', '0123456789']
fig, axes = plt.subplots(len(tests), 1, figsize=(14, 3*len(tests)))

for ax, word in zip(axes, tests):
    _, page = generate_word(model, word, device=device, noise_scale=0.018)
    ax.imshow(page, cmap='gray')
    ax.axis('off')
    ax.set_title(f'"{word}"', fontsize=12)

plt.tight_layout()
plt.savefig(f'{REPO_PATH}/outputs/demo.png', dpi=120, bbox_inches='tight')
plt.show()
print('Demo saved to outputs/demo.png')